Решим задачу регрессии для SI

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel(
    "Данные_для_курсовои_Классическое_МО.xlsx"
)
df = df.drop(columns=["Unnamed: 0"])
df = df.fillna(df.median())

In [2]:
targets = ["IC50, mM", "CC50, mM", "SI"]
X = df.drop(columns=targets)

y = np.log1p(df["SI"])

In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

In [4]:
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
from sklearn.linear_model import LinearRegression

lr = LinearRegression() #нету значимых гиперпараметров
lr.fit(X_train, y_train)

pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))

mae = mean_absolute_error(y_test, pred)

r2 = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 1.467925796673171
MAE: 1.0805096966283267
R2: 0.10958693102951977


Линейная регрессия использована в качестве базовой модели. Полученное значение R2 = 0.10 говорит о том что данная модель не подходит для такого вида данных


In [5]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=700) #оптимальное число деревьев для обучения модели, большее кол-во слабо влияет на метрики
rf.fit(X_train, y_train)

pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))

mae = mean_absolute_error(y_test, pred)

r2 = r2_score(y_test, pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 1.281620795784712
MAE: 0.9270378700996855
R2: 0.32126161323534197


Случайный лес напротив показывает более высокие значения R2 = 0.32

Посмотрим также градиентный бустинг

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42, max_depth=10, min_samples_split=5) #глубина оптимальная, нету недообучения и риска переобучения
                                                                                    #min_samples_split делает деревья менее сложными борется с переобучением

gb.fit(X_train, y_train)

pred = gb.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Gradient Boosting")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Gradient Boosting
RMSE: 1.3083977552007298
MAE: 0.9629185547473772
R2: 0.2926035091628705


Итоговые значения оказались также хуже чем по модели случайного леса. Модель случайного леса для задачи регрессии оказалась также самой лучшей

Используем выделенные наиболее важные признаки дескрипторы который выделили в EDA и которые оказывают наибольшее влияние на показатели эффективности. Проверим насколько хорошо отдельно эти признаки описывают основную полезную информацию

In [6]:
selected_features = [
    "VSA_EState8",
    "VSA_EState6",
    "VSA_EState4",
    "MolLogP",
    "qed",
    "FractionCSP3",
    "NumSaturatedHeterocycles",
    "NumSaturatedCarbocycles",
    "NumAliphaticCarbocycles"
]

X_small = df[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_small,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
rf = RandomForestRegressor(n_estimators=700)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

In [8]:
rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("Random Forest (selected features)")
print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

Random Forest (selected features)
RMSE: 1.3052118158509884
MAE: 0.9408676886328047
R2: 0.2960443258534673


Мы сократили число признаков с 211 всего до 9, при этом качество R2 упало лишь чуть более чем на 2%, что подтверждает высокую информативность наших 9 признаков, выявленных ранее на этапе EDA.

Во всех трёх задачах регрессии наилучшие результаты показал Random Forest, что свидетельствует о наличии сложных нелинейных зависимостей между молекулярными дескрипторами и исследуемыми показателями.

Однако в связи с невысоким значением R2 регрессионные модели пока не кажутся перспективными